# Week 11 — Test-set showcase, empirical target (11e)

**Held-out verdict on the 6 test hemispheric cycles for the empirical-distribution
models** trained by `11c_train_empirical.ipynb`. Same figure grammar as
`11c_test_showcase.ipynb` (which showcases the *residual*-target E0), so the two
notebooks can be read side by side:

| row | source |
|-----|--------|
| Empirical | observed sunspot-emergence histograms (the reference) |
| SARGE | independent physics baseline — its own universal μ(τ) + σ(A), **fit on the training split** |
| Classical | the ButterflAI parametric model (`official_model.npz`) |
| AI (empirical) | the logit/softmax diffusion model (`ckpt_emp_<name>.ckpt`), sampled directly |

The figure is a **5 × 6 grid** — rows = the four sources plus one single-draw
realization of the AI model; columns = the 6 test hemicycles (12N, 13S, 15S, 19N,
22S, 24S). Every non-empirical panel is annotated with **NLL, EMD, energy,
crps_mu**; a pooled **test scorecard** and a **held-out ranking of every trained
empirical variant** close the notebook.

## What differs from 11c

The empirical-target model predicts the *whole* per-window distribution in
standardized logit space and decodes through a softmax, so:

- there is **no `+ p_classical` pedestal** — generated densities go straight into
  `assemble_butterfly_direct` instead of `assemble_butterfly`;
- every sampled density is non-negative and integrates to 1 by construction, so
  the renormalized NLL used here (`hard_nll_direct` on renormalized densities)
  equals the raw one. The renormalization is kept anyway so the AI row's NLL is
  *definitionally* the same metric as the classical and SARGE rows' — the numbers
  are directly comparable to 11c's.

The hard gate (`μ(τ) ≤ μ₀(A)`, from the classical model) is identical in both
notebooks, so the scored window set matches column for column.

Only SARGE's scalar `k` and its μ(τ) are fit here — on the **training split only**,
so nothing is tuned on the held-out test data. The diffusion weights are loaded as-is.

> The v2 conditioning parquet (`weeks/week_10`) ships train+val only; the test rows are
> re-derived **in memory** from the v1 parquet with the same (train-fit) augmentation, so
> the committed artifacts are never modified.

In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
from scipy.optimize import curve_fit, minimize_scalar
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat

import pytorch_lightning as pl

In [ ]:
# Bootstrap sys.path and locate artifacts. (Same resolver as 11c/11d; see 11b's
# comments for why week_10 is forced ahead of the week_09 stub.)
import os, sys

_week09_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))
_week10_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_10"))
_week11_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_11"))
for _p in (_week09_dir, _week11_dir, _week10_dir):
    if _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)
sys.modules.pop("conditioned_infrastructure", None)

from conditioned_infrastructure import find_week10_artifacts
paths = find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
# Repoint the modules to the week_11 copies (this notebook's API home).
if _week11_dir in sys.path:
    sys.path.remove(_week11_dir)
sys.path.insert(0, _week11_dir)
sys.modules.pop("conditioned_infrastructure", None)
sys.modules.pop("empirical_infrastructure", None)

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    block_cond_concat,
    k_run_combined,
    build_experiment_registry,
    resolve_experiment_stems,
)
from empirical_infrastructure import (
    EmpiricalDistributionDataset,
    ExtendedConditionalEmpiricalDiffusionLightning,
    load_trained_empirical_experiment,
    sample_empirical_extended,
    discover_emp_experiment_checkpoints,
    hard_nll_direct,
    normalize_densities,
    assemble_butterfly_direct,
    EMP_CKPT_PREFIX,
)
from evaluation import (
    build_eval_hemicycles,
    hard_nll_combined_normalized,
    assemble_butterfly,
    butterfly_physical_checks,
    distributional_scorecard,
)
from butterflAI_model import ButterflAIModel

import empirical_infrastructure as _ei
print(f"using empirical_infrastructure from: {_ei.__file__}")

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# THIS notebook is the test-set showcase: load the v1 parquet WITH the test split.
windows_v1_full = pd.read_parquet(paths["parquet_v1"])
print(f"v1 parquet (all splits): {len(windows_v1_full)} rows")
print(f"  splits: {windows_v1_full['split'].value_counts().sort_index().to_dict()}")
print(f"device : {device}")

---
## Step 1 — Rebuild the v2 conditioning parquet **including test**

Identical to 11c. The committed v2 parquet holds train+val only, so we re-run the exact
augmentation from `11_00_build_v2.ipynb` (Tasks 60–62) on the full v1 table. Every
normalization constant (`cycle_norm` range, `opp_*` / `area_lag*` imputation means) is fit
on the **train split only** — so the train/val columns come out bit-identical to the
committed parquet and the test rows get leak-free conditioning. Nothing is written to disk.

In [ ]:
# --- Task 60: cycle_norm + hemi_id (train-set cycle range) ---
windows_aug = windows_v1_full.copy()
_train_cycles = windows_aug.loc[windows_aug["split"] == "train", "cycle"]
_cmin, _cmax  = _train_cycles.min(), _train_cycles.max()
windows_aug["cycle_norm"] = 2.0 * (windows_aug["cycle"] - _cmin) / (_cmax - _cmin) - 1.0
windows_aug["hemi_id"] = np.where(windows_aug["hemisphere"] == "north", 1.0, -1.0)

# --- Task 61: contemporaneous opposite-hemisphere summaries ---
_t0 = {(int(c), str(h)): float(classical.lookup_t0(int(c), h))
       for (c, h) in classical.known_hemicycles()}

def _to_year(row):
    key = (int(row["cycle"]), str(row["hemisphere"]))
    return float(row["tau_center"]) + _t0[key] if key in _t0 else np.nan

windows_aug["_year_center"] = windows_aug.apply(_to_year, axis=1)
_flip = {"north": "south", "south": "north"}
# Opposite-hemisphere SOURCE pool = train+val only. This keeps the train/val
# opp_* columns bit-identical to the committed v2 (no test row ever informs a
# train/val feature) while still giving every test row a real opposite-
# hemisphere match (each test hemicycle's opposite hemisphere is in train/val).
_right = (windows_aug.loc[windows_aug["split"].isin(["train", "val"]),
                          ["cycle", "_year_center", "hemisphere",
                           "area_smoothed", "mu_universal", "amplitude"]]
          .assign(hemisphere=lambda d: d["hemisphere"].map(_flip))
          .rename(columns={"area_smoothed": "opp_area_smoothed",
                           "mu_universal":  "opp_mu_universal",
                           "amplitude":     "opp_amplitude"}))
_TOL = 0.4
_left = (windows_aug.reset_index().rename(columns={"index": "_rowid"})
         .sort_values("_year_center"))
_right = _right.sort_values("_year_center")
_merged = pd.merge_asof(_left, _right, on="_year_center", by=["cycle", "hemisphere"],
                        direction="nearest", tolerance=_TOL)
windows_aug = (_merged.sort_values("_rowid")
               .drop(columns=["_rowid", "_year_center"]).reset_index(drop=True))
windows_aug["opp_valid"] = windows_aug["opp_area_smoothed"].notna().astype(np.float32)
_opp_cols = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"]
_train_mask = (windows_aug["split"] == "train") & (windows_aug["opp_valid"] == 1.0)
for _c in _opp_cols:
    windows_aug[_c] = windows_aug[_c].fillna(windows_aug.loc[_train_mask, _c].mean())

# --- Task 62: smoothed-area trajectory (K lagged values) ---
K_LAGS = 4
_sorted = windows_aug.sort_values(["cycle", "hemisphere", "tau_center"])
_lag_cols = [f"area_lag{k}" for k in range(1, K_LAGS + 1)]
for k, col in enumerate(_lag_cols, start=1):
    _sorted[col] = _sorted.groupby(["cycle", "hemisphere"])["area_smoothed"].shift(k)
_sorted["traj_valid"] = (~_sorted[_lag_cols].isna().any(axis=1)).astype(np.float32)
_train_mask = (_sorted["split"] == "train") & (_sorted["traj_valid"] == 1.0)
for col in _lag_cols:
    _sorted[col] = _sorted[col].fillna(_sorted.loc[_train_mask, col].mean())
windows_v2_full = _sorted.sort_index()

# Sanity: the train/val rows must match the committed v2 parquet bit-for-bit.
_committed = pd.read_parquet(PARQUET_V2)
_tv = windows_v2_full[windows_v2_full["split"].isin(["train", "val"])].reset_index(drop=True)
assert len(_tv) == len(_committed), (len(_tv), len(_committed))
_NEW = ["cycle_norm", "hemi_id", *_opp_cols, *_lag_cols]
for c in _NEW:
    a = _tv.sort_values(["cycle", "hemisphere", "tau_center"])[c].to_numpy()
    b = _committed.sort_values(["cycle", "hemisphere", "tau_center"])[c].to_numpy()
    assert np.allclose(a, b, equal_nan=True), f"drift vs committed v2 in {c}"
print("v2(full) rows:", len(windows_v2_full),
      "| splits:", windows_v2_full["split"].value_counts().sort_index().to_dict())
print("train/val columns reproduce the committed v2 parquet exactly.")

---
## Step 2 — Build the 6 test hemicycles

`build_eval_hemicycles` re-windows the raw catalog 6-monthly and tags each window with its
v2 conditioning superset. We ask for `splits=("test",)` so only the held-out hemicycles are
built. Same builder, same arguments as 11c — so the two showcases score the same blocks.

In [ ]:
hemicycles, GROUP_COLS = build_eval_hemicycles(
    raw_csv_path=paths["raw_csv"],
    windows_v2=windows_v2_full,
    classical=classical,
    splits=("test",),
)
test_hcs = sorted([hc for hc in hemicycles if hc["split"] == "test"],
                  key=lambda h: (h["cycle"], h["hemisphere"]))

def tag_of(hc):
    return f"{hc['cycle']:02d}{hc['hemisphere'][0].upper()}"

tags = [tag_of(hc) for hc in test_hcs]
print(f"test hemicycles ({len(test_hcs)}): {tags}")
print(f"windows per hemicycle: {[len(hc['blocks']) for hc in test_hcs]}")
assert len(test_hcs) == 6, "expected exactly 6 test hemicycles"
assert set(tags) == {"12N", "13S", "15S", "19N", "22S", "24S"}, tags

---
## Step 3 — Experiment registry and the showcased variant

`_SPECS_EMP` must **mirror** the list in `11c_train_empirical.ipynb` /
`11d_evaluate_empirical.ipynb`: the loader needs each checkpoint's `consumed_keys`,
architecture and smoothing knobs to rebuild the dataset identically. If you add a variant
to the training notebook, copy its spec here too — the run name is generated from
the spec, so mirroring the specs is enough.

`SHOWCASE_NAME` picks the variant the figures are built from. **Set it from your
val results in 11d, not from the test table at the bottom of this notebook** — picking the
winner by test NLL would be test-set model selection and would invalidate the held-out
claim. It defaults to `base_cat`, the baseline, matching 11c's choice.

`SHOWCASE_W` is the classifier-free guidance weight. Non-CFG variants
(`cond_dropout_p == 0`) have no trained null embedding, so guidance is meaningless for them
and the loader below forces `w = 0`.

In [ ]:
# Experiment specs — mirror of 11c_train_empirical / 11d_evaluate_empirical.
# (max_epochs is irrelevant at load time; kept so the spec lists stay
# copy-pasteable between the three notebooks.)
#
# Run names are GENERATED from the specs by canonical_experiment_name(), not
# typed by hand, so a name can never disagree with the config it labels:
#
#     <cond-set>_<arch>[_four][_guid][_h###][_L#]
#
#   cond-set : "base" plus added groups in the fixed order hemi < opp < traj,
#              joined by "+"  ("trajv" = traj group + its validity mask)
#   arch     : "cat" (concat) or "film" — ALWAYS stated
#   four     : fourier=True          guid : cond_dropout_p > 0
#   h###/L#  : appended only when capacity differs from _BASE_TEMPLATE
#
# Checkpoints land at ckpt_emp_<name>.ckpt (EMP_CKPT_PREFIX), so these share
# the residual family's names without colliding with its files.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     5000,
    "lr":             1e-3,
    "batch_size":     64,
    # Empirical-target specific knobs.
    "lambda_kl":      0.1,
    "alpha_smooth":   1.0,
    "n_pseudo_obs":   30.0,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

_SPECS_EMP = [
    # ── Base conditioning (4-D cond) × mechanism ────────────────────────────
    # The full 2×2×2 over arch, Fourier lifting and guidance, holding the
    # information content fixed: how much comes from mechanism alone?
    _spec(),                                                   # base_cat  (baseline)
    _spec(arch="film"),                                        # base_film
    _spec(fourier=True),                                       # base_cat_four
    _spec(arch="film", fourier=True),                          # base_film_four
    _spec(cond_dropout_p=0.1),                                 # base_cat_guid
    _spec(arch="film", cond_dropout_p=0.1),                    # base_film_guid
    _spec(fourier=True, cond_dropout_p=0.1),                   # base_cat_four_guid
    _spec(arch="film", fourier=True, cond_dropout_p=0.1),      # base_film_four_guid

    # ── Level 1 — one new cond group at a time, concat arch ─────────────────
    _spec(consumed_keys=["cond_base", "cond_cyclehemi"],       # base+hemi_cat
          groups=["base", "cyclehemi"]),
    _spec(consumed_keys=["cond_base", "cond_opp"],             # base+opp_cat
          groups=["base", "opp"]),
    _spec(consumed_keys=["cond_base", "cond_traj"],            # base+traj_cat
          groups=["base", "traj"]),

    # ── Levels 3–5 — best L1 cond group (opp) × mechanism ───────────────────
    _spec(arch="film",                                         # base+opp_film
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"]),
    _spec(arch="film",                                         # base+opp_film_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          cond_dropout_p=0.1),
    _spec(arch="film",                                         # base+opp_film_four
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True),
    _spec(arch="film",                                         # base+opp_film_four_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True,
          cond_dropout_p=0.1),

    # ── Trajectory conditioning, and the opp × traj pair ────────────────────
    _spec(arch="film",                                         # base+traj_film_four
          consumed_keys=["cond_base", "cond_traj"],
          groups=["base", "traj"],
          fourier=True),
    _spec(arch="film",                                         # base+opp+traj_film_four
          consumed_keys=["cond_base", "cond_opp", "cond_traj"],
          groups=["base", "opp", "traj"],
          fourier=True),
    # Same as base+traj_film_four plus the window-validity mask — the "v".
    _spec(arch="film",                                         # base+trajv_film_four
          consumed_keys=["cond_base", "cond_traj", "cond_traj_valid"],
          groups=["base", "traj"],
          fourier=True),
]

# Keys the registry by canonical name; raises if two specs collide, which
# catches both duplicates and knobs the naming scheme doesn't yet encode.
EXPERIMENTS_EMP = build_experiment_registry(_SPECS_EMP, _BASE_TEMPLATE)

# Every ckpt_emp_*.ckpt that maps to a spec — canonical stems and the
# pre-rename E<n>metrics ones alike. EMP_STEM translates a canonical name to
# the file it actually lives in.
_ckpts_emp = discover_emp_experiment_checkpoints(CKPT_DIR)
_known_emp, _unknown = resolve_experiment_stems(_ckpts_emp, EXPERIMENTS_EMP)
KNOWN_EMP = [name for name, _ in _known_emp]
EMP_STEM  = dict(_known_emp)
if _unknown:
    print(f"WARNING: checkpoints with no EXPERIMENTS_EMP spec (skipped): {_unknown}")
print(f"empirical checkpoints available: {KNOWN_EMP}")


In [ ]:
# Which variant do the figures showcase? Choose it from the VAL results in
# 11d_evaluate_empirical.ipynb — never from this notebook's test table.
SHOWCASE_NAME = "base_cat"
SHOWCASE_W    = 0.0          # CFG weight; forced to 0 for non-CFG variants

assert SHOWCASE_NAME in EXPERIMENTS_EMP, \
    f"{SHOWCASE_NAME} has no EXPERIMENTS_EMP spec"
assert SHOWCASE_NAME in KNOWN_EMP, \
    f"no checkpoint for {SHOWCASE_NAME} in {CKPT_DIR} "\
    f"(looked for {EMP_CKPT_PREFIX}{SHOWCASE_NAME}.ckpt and its pre-rename stem)"

EMP_CFG = EXPERIMENTS_EMP[SHOWCASE_NAME]
EMP_W   = float(SHOWCASE_W) if EMP_CFG.get("cond_dropout_p", 0.0) > 0.0 else 0.0
if EMP_W != float(SHOWCASE_W):
    print(f"note: {SHOWCASE_NAME} was trained without cond dropout — "
          f"guidance forced to w=0 (no null embedding to guide against)")

# The datasets are rebuilt from the train/val rows only (the loader filters by
# `split`), so passing the test-bearing frame changes nothing about the
# standardization the model was trained with.
emp_lit, emp_train_ds, emp_val_ds, emp_total_dim = load_trained_empirical_experiment(
    name=EMP_STEM[SHOWCASE_NAME], cfg=EMP_CFG, windows_aug=windows_v2_full,
    classical=classical, bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
    alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
)
emp_lit.eval()
print(f"loaded {SHOWCASE_NAME}: arch={EMP_CFG['arch']}, cond={EMP_CFG['consumed_keys']}, "
      f"cond_dim={emp_total_dim}, guidance w={EMP_W}")

---
## Step 4 — Fit SARGE on the training split

Unchanged from 11c. SARGE is a fully independent physics baseline. Following
`week_06/fit_sarge_sigma_width.py` we fit **only two things on the training hemicycles**:
the universal mean path μ(τ)=a·exp(−τ/b) and the single spread scale `k` in
σ_λ(A) = 1.5° + 3.8°·[1 − exp(−A/k)]. The 15°-crossing t0 is computed for **all**
hemicycles (data-driven, so we can place the test bands), and the global smoothed
total-area series drives σ per window. Nothing about SARGE is tuned on val or test.

In [ ]:
# --- SARGE constants (paper-prescribed; only k is fit) ---
SARGE_SIGMA_MIN, SARGE_SIGMA_AMP = 1.5, 3.8
AREA_THRESHOLD, MIN_CYCLE = 30, 12
N_BINS, MIN_LATS_BIN, MIN_LATS_YEAR, SMOOTH_WIN = 20, 10, 5, 365

# train hemicycle set from the diffusion split -> SARGE never sees val/test.
_hc_split = (windows_v2_full.groupby(["cycle", "hemisphere"])["split"]
             .agg(lambda s: s.iloc[0]).to_dict())
train_hc = {(int(c), str(h)) for (c, h), s in _hc_split.items() if s == "train"}
print(f"SARGE fit on {len(train_hc)} train hemicycles")

# raw catalog
sdf = pd.read_csv(paths["raw_csv"], parse_dates=[[0, 1, 2]], keep_date_col=False)
sdf.rename(columns={"year_month_day": "date"}, inplace=True)
sdf = sdf[sdf["latitude"].notna()].copy()
sdf["hemisphere"]   = np.where(sdf["latitude"] >= 0, "north", "south")
sdf["abs_latitude"] = sdf["latitude"].abs()
sdf["year"]         = sdf["date"].dt.year
sdf["decimal_year"] = sdf["date"].dt.year + sdf["date"].dt.dayofyear / 365.25
sdf = sdf[sdf["correctedArea"] > AREA_THRESHOLD].copy()
sdf = sdf.dropna(subset=["CYCLE"]).copy()
sdf["CYCLE"] = sdf["CYCLE"].astype(int)

def exp_decay(tau, a, b):
    return a * np.exp(-tau / b)

# 15-degree crossing t0 for ALL hemicycles (test included -> placeable bands)
def _t0_15(group):
    ym = group.groupby("year")["abs_latitude"].mean().sort_index()
    years, means = ym.index.values, ym.values
    below = means < 15.0
    if not below.any() or below.all():
        return None
    i1 = int(np.argmax(below))
    if i1 == 0:
        return None
    y0, m0 = years[i1 - 1], means[i1 - 1]
    y1, m1 = years[i1], means[i1]
    return float(y0 + (15.0 - m0) / (m1 - m0))

t0_lookup = {}
for (cyc, hemi), g in sdf.groupby(["CYCLE", "hemisphere"]):
    v = _t0_15(g)
    if v is not None:
        t0_lookup[(int(cyc), hemi)] = v
sdf["t0"]  = sdf.apply(lambda r: t0_lookup.get((int(r["CYCLE"]), r["hemisphere"]), np.nan), axis=1)
sdf["tau"] = sdf["decimal_year"] - sdf["t0"]

# universal mu(tau): fit on TRAIN hemicycle bins only; refine t0 for all.
cycles = [c for c in sorted(sdf["CYCLE"].dropna().unique()) if c >= MIN_CYCLE]
all_t, all_m, hemibins = [], [], {}
for cyc in cycles:
    for hemi in ["north", "south"]:
        key = (int(cyc), hemi)
        if key not in t0_lookup:
            continue
        ds = sdf[(sdf["CYCLE"] == cyc) & (sdf["hemisphere"] == hemi) & sdf["tau"].notna()]
        if len(ds) < 50:
            continue
        bins = np.linspace(ds["tau"].min(), ds["tau"].max(), N_BINS + 1)
        bc = 0.5 * (bins[:-1] + bins[1:])
        bt, bm = [], []
        for i in range(N_BINS):
            lb = ds.loc[(ds["tau"] >= bins[i]) & (ds["tau"] < bins[i + 1]), "abs_latitude"].values
            if len(lb) < MIN_LATS_BIN:
                continue
            mu_f, _ = sp_norm.fit(lb)
            bt.append(bc[i]); bm.append(mu_f)
        if len(bt) < 5:
            continue
        bt, bm = np.array(bt), np.array(bm)
        hemibins[key] = (bt, bm)
        if key in train_hc:                      # TRAIN-only universal fit
            all_t.extend(bt); all_m.extend(bm)

(a_mu_univ, b_mu_univ), _ = curve_fit(exp_decay, all_t, all_m, p0=[15.0, 5.0])
print(f"universal mu(tau): a={a_mu_univ:.3f} deg, b={b_mu_univ:.3f} yr  (train-only)")

t0_refined = {}
for key, (bt, bm) in hemibins.items():
    def _res(dt, _bt=bt, _bm=bm):
        return np.sum((_bm - exp_decay(_bt - dt, a_mu_univ, b_mu_univ)) ** 2)
    t0_refined[key] = t0_lookup[key] + minimize_scalar(_res, bounds=(-4, 4),
                                                       method="bounded").x

# global smoothed total area (both hemispheres; physical series, not split-specific)
_dn = sdf[sdf["hemisphere"] == "north"].groupby("date")["correctedArea"].sum()
_dsouth = sdf[sdf["hemisphere"] == "south"].groupby("date")["correctedArea"].sum()
_dr = pd.date_range(min(_dn.index.min(), _dsouth.index.min()),
                    max(_dn.index.max(), _dsouth.index.max()), freq="D")
_dn = _dn.reindex(_dr, fill_value=0); _dsouth = _dsouth.reindex(_dr, fill_value=0)
smooth_total = (_dn + _dsouth).rolling(SMOOTH_WIN, center=True,
                                       min_periods=SMOOTH_WIN // 3).mean()

def A_total_at(decimal_year):
    """Global smoothed total area (MSH) nearest to a decimal year, or nan."""
    yr = int(np.floor(decimal_year))
    d  = pd.Timestamp(year=yr, month=1, day=1) + pd.Timedelta(days=(decimal_year - yr) * 365.25)
    idx = smooth_total.index.get_indexer([d], method="nearest")[0]
    v = float(smooth_total.iloc[idx])
    return v if np.isfinite(v) and v > 0 else np.nan

# k fit: TRAIN hemicycles only (min mean per-year NLL)
def sarge_sigma(A, k):
    return SARGE_SIGMA_MIN + SARGE_SIGMA_AMP * (1.0 - np.exp(-A / k))

_cache = []   # (lats, mu, A_total) per train (cycle, hemisphere, year)
for cyc in cycles:
    for hemi in ["north", "south"]:
        key = (int(cyc), hemi)
        if key not in t0_refined or key not in train_hc:
            continue
        dch = sdf[(sdf["CYCLE"] == cyc) & (sdf["hemisphere"] == hemi) & sdf["tau"].notna()]
        for yr in sorted(dch["year"].unique()):
            lats = dch.loc[dch["year"] == yr, "abs_latitude"].values
            if len(lats) < MIN_LATS_YEAR:
                continue
            mu = float(exp_decay((yr + 0.5) - t0_refined[key], a_mu_univ, b_mu_univ))
            A  = A_total_at(yr + 0.5)
            if np.isfinite(A):
                _cache.append((lats, mu, A))

def _nll_k(k):
    if k <= 0:
        return 1e10
    tot = 0.0
    for lats, mu, A in _cache:
        tot -= sp_norm.logpdf(lats, loc=mu, scale=sarge_sigma(A, k)).mean()
    return tot / len(_cache)

_kres = minimize_scalar(_nll_k, bounds=(0.5, 2000.0), method="bounded", options={"xatol": 0.1})
k_fit = float(_kres.x)
print(f"SARGE k (train-only): {k_fit:.1f} MSH  |  mean NLL={_kres.fun:.4f} nats/yr  "
      f"|  n_entries={len(_cache)}")

# generative handles used by the butterfly assembly below
def sarge_mu(tau):
    return float(exp_decay(tau, a_mu_univ, b_mu_univ))

def sarge_t0(cyc, hemi):
    return t0_refined.get((int(cyc), str(hemi)), float(classical.lookup_t0(int(cyc), str(hemi))))

def sarge_bin_density(decimal_year, cyc, hemi):
    """SARGE Gaussian density on BIN_CENTERS for one window."""
    mu  = sarge_mu(decimal_year - sarge_t0(cyc, hemi))
    A   = A_total_at(decimal_year)
    sig = sarge_sigma(A, k_fit) if np.isfinite(A) else SARGE_SIGMA_MIN + SARGE_SIGMA_AMP
    return sp_norm.pdf(BIN_CENTERS, loc=mu, scale=sig), mu, sig

---
## Step 5 — Assemble butterfly maps and per-panel metrics

For each test hemicycle we take the classical **hard-gate** window set (windows where the
classical mean latitude is still active, `μ(τ) ≤ μ₀`) as the shared time axis, so all four
rows in a column line up. Per window we build a 15-bin density for each model:

- **Empirical** — `np.histogram` of the observed \|latitude\|.
- **Classical** — the classical Gaussian at the bin centers.
- **AI (empirical)** — the mean over K = 1000 softmax-decoded samples. A mean of densities
  on the simplex is itself a density, so this row is a proper distribution.
- **SARGE** — `N(μ_SARGE(τ), σ_SARGE(A))` at the bin centers.

Metrics per panel (SARGE / Classical / AI; the empirical panel is the reference and carries
no metrics): **renormalized NLL** (proper-density NLL over the 15-bin support), **EMD**,
**energy**, and **crps_mu** via `distributional_scorecard`.

The AI NLL goes through `hard_nll_direct` on renormalized densities — the same
"divide by the mass on the 15-bin support" convention `hard_nll_combined_normalized`
applies to the classical row, so the two columns of the scorecard are the same metric.
A simplex sanity check runs on every sampled batch before anything is scored.

In [ ]:
K = 1000
AI_ROW     = "AI (empirical)"
MODEL_ROWS = ["Empirical", "SARGE", "Classical", AI_ROW]

def _simplex_sanity(samples, bin_width=BIN_WIDTH):
    """Assert non-negativity + integrate-to-1 on every (15,) density vector."""
    nonneg = float(np.min(samples))
    mass   = samples.sum(axis=-1) * bin_width
    err    = float(np.abs(mass - 1.0).max())
    assert nonneg >= -1e-6, f"simplex check failed: min density = {nonneg}"
    assert err     <  1e-4, f"simplex check failed: max |Σp·Δℓ - 1| = {err}"
    return {"min_density": nonneg, "max_mass_err": err}

def _classical_bin_density(hc, blk):
    # hc["amplitude"] is MSH/1000 -- the unit classical.sigma / mu_0 were fit
    # on. hc["amplitude_msh"] is the raw-MSH twin, for labels only.
    A  = hc["amplitude"]
    mu = float(classical.mu(blk["tau"]))
    sg = float(classical.sigma(mu, A))
    return sp_norm.pdf(BIN_CENTERS, loc=mu, scale=sg)

def nll_norm_direct(model, hcs, dens_by_block):
    """Renormalized NLL for a directly-generated density, in the
    (model, hcs, densities_by_block) shape k_run_combined expects. The
    renormalization is a no-op on softmax output — it is kept so this is
    definitionally the same metric as hard_nll_combined_normalized."""
    return hard_nll_direct(model, hcs,
                           normalize_densities(dens_by_block, BIN_WIDTH),
                           bin_width=BIN_WIDTH)

def _norm_gauss_nll(hc, gated_blocks, dens_fn):
    """Renormalized (proper-density) NLL for a per-window bin-density model.

    dens_fn(blk) -> (15,) density on BIN_CENTERS. Scores the observed lats at
    their bin, divided by the density's mass on the 15-bin support (+log Z),
    matching hard_nll_combined_normalized's convention.
    """
    tot, inc, eps = 0.0, 0, 1e-6
    for blk in gated_blocks:
        dens = np.maximum(eps, dens_fn(blk))
        Z = max(eps, float((dens * BIN_WIDTH).sum()))
        bin_ix = np.clip(np.floor(blk["lats"] / BIN_WIDTH).astype(int), 0, 14)
        ll = np.log(dens[bin_ix]).mean()
        if np.isfinite(ll):
            tot -= (ll - np.log(Z)); inc += 1
    return tot / inc if inc else float("nan")

def build_hc(hc):
    """Return maps_by_row, per-panel metrics, and the density dicts for pooling."""
    cyc, hemi = hc["cycle"], hc["hemisphere"]

    # K decoded densities per block. No classical pedestal: the model predicts
    # the whole distribution, so `samp` IS the density.
    keys, cond = block_cond_concat([hc], emp_lit, EMP_CFG, emp_train_ds)
    torch.manual_seed(0)
    with torch.no_grad():
        samp = sample_empirical_extended(
            emp_lit, cond.repeat_interleave(K, dim=0), guidance_w=EMP_W,
            bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy().reshape(len(keys), K, 15)
    _simplex_sanity(samp)
    samp_by_key = {k: samp[i] for i, k in enumerate(keys)}
    mean_dens   = {k: samp[i].mean(axis=0) for i, k in enumerate(keys)}

    # Classical hard gate defines the shared window set (t_ref). Both assembly
    # helpers apply the identical gate, so the AI and classical rows align.
    m_ai, t_ref = assemble_butterfly_direct(classical, hc, mean_dens, BIN_CENTERS)
    zero = {(cyc, hemi, float(t)): np.zeros(15) for t in t_ref}
    m_cl, _ = assemble_butterfly(classical, hc, zero, BIN_CENTERS)

    blk_by_t = {float(b["center_decimal"]): b for b in hc["blocks"]}
    gated_blocks = [blk_by_t[float(t)] for t in t_ref]

    emp = np.array([np.histogram(b["lats"], bins=LAT_BINS, density=True)[0]
                    for b in gated_blocks]) if len(t_ref) else np.empty((0, 15))
    m_sarge = np.array([sarge_bin_density(float(t), cyc, hemi)[0] for t in t_ref]) \
        if len(t_ref) else np.empty((0, 15))

    maps = {"Empirical": (emp, t_ref), "SARGE": (m_sarge, t_ref),
            "Classical": (m_cl, t_ref), AI_ROW: (m_ai, t_ref)}

    # --- density dicts for the distributional metrics (scorecard convention) ---
    emp_dens, tau_by, cl_dens, sarge_dens, ai_dens = {}, {}, {}, {}, {}
    for t, blk in zip(t_ref, gated_blocks):
        key = (cyc, hemi, float(t))
        par = _classical_bin_density(hc, blk)
        emp_dens[key]   = np.histogram(blk["lats"], bins=LAT_BINS, density=True)[0]
        tau_by[key]     = blk["tau"]
        cl_dens[key]    = par[None, :]
        sarge_dens[key] = sarge_bin_density(float(t), cyc, hemi)[0][None, :]
        ai_dens[key]    = samp_by_key[key]                        # (K, 15)

    # --- per-panel metrics ---
    def _dist(md_):
        r = distributional_scorecard(md_, emp_dens, BIN_CENTERS, BIN_WIDTH, tau_by)
        return {m: r[m] for m in ("emd", "energy", "crps_mu")}

    nll_cl = hard_nll_combined_normalized(classical, [hc], zero, BIN_CENTERS,
                                          bin_width=BIN_WIDTH)[0]
    nll_ai = k_run_combined(nll_norm_direct, classical, [hc], keys, samp)[0].mean()
    nll_sarge = _norm_gauss_nll(
        hc, gated_blocks, lambda b: sarge_bin_density(float(b["center_decimal"]), cyc, hemi)[0])

    metrics = {
        "SARGE":     {"nll": nll_sarge, **_dist(sarge_dens)},
        "Classical": {"nll": nll_cl,    **_dist(cl_dens)},
        AI_ROW:      {"nll": float(nll_ai), **_dist(ai_dens)},
    }
    pooled = {"emp": emp_dens, "tau": tau_by,
              "SARGE": sarge_dens, "Classical": cl_dens, AI_ROW: ai_dens,
              "ai_keys": keys, "ai_samp": samp}
    return maps, metrics, pooled

def single_emp_map(hc):
    """One fresh reverse-diffusion realization (a single K=1 draw per window)
    -> (density_map, times). Deliberately *unseeded* so each call yields a
    different sample; the figure cells seed from entropy before calling, so
    re-running them rolls a new realization."""
    keys, cond = block_cond_concat([hc], emp_lit, EMP_CFG, emp_train_ds)
    with torch.no_grad():
        s = sample_empirical_extended(emp_lit, cond, guidance_w=EMP_W,
                                      bin_width=BIN_WIDTH,
                                      device=device).cpu().numpy()
    dens = {k: s[i] for i, k in enumerate(keys)}
    return assemble_butterfly_direct(classical, hc, dens, BIN_CENTERS)

maps_by_hc, metrics_by_hc, pooled_by_hc = {}, {}, {}
for hc in test_hcs:
    m, mt, pl = build_hc(hc)
    tg = tag_of(hc)
    maps_by_hc[tg], metrics_by_hc[tg], pooled_by_hc[tg] = m, mt, pl
    print(f"{tg}: {len(m['Classical'][1])} gated windows  "
          f"| AI NLL={mt[AI_ROW]['nll']:.3f}  EMD={mt[AI_ROW]['emd']:.3f}")

---
## The figure — empirical vs SARGE vs classical vs AI (empirical target) on the test set

Rows = the four sources plus one single-draw AI realization; columns = the 6 test
hemicycles. Cyan dotted lines mark the Spörer band (5°–40°). Every non-empirical panel
shows its NLL / EMD / energy / crps_mu.

In [ ]:
import time
from matplotlib.ticker import MaxNLocator
vmax = 0.15
SINGLE_ROW = f"{AI_ROW} (single draw)"
YLAB = {"Empirical": "Empirical", "SARGE": "SARGE", "Classical": "Classical",
        AI_ROW: "AI (empirical)",
        SINGLE_ROW: "AI (empirical)\n(single draw)"}

# Roll ONE fresh reverse-diffusion realization per window (bottom row). Seeded
# from entropy so re-running THIS cell resamples a different draw every time.
torch.manual_seed(time.time_ns() % (2**31))
single_maps = {tag_of(hc): single_emp_map(hc) for hc in test_hcs}

fig_rows = MODEL_ROWS + [SINGLE_ROW]
nrows, ncols = len(fig_rows), len(test_hcs)
# Column widths proportional to each hemicycle's duration (years) so every
# x axis shares one year scale -> short cycles get thinner columns.
col_spans = [max(float(np.ptp(maps_by_hc[tag_of(hc)]["Classical"][1])), 0.5)
             if maps_by_hc[tag_of(hc)]["Classical"][1].size else 1.0
             for hc in test_hcs]
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(0.55 * sum(col_spans) + 2.0, 2.7 * nrows),
                         sharex="col", sharey=True, squeeze=False,
                         gridspec_kw={"width_ratios": col_spans},
                         layout="constrained")

im = None
for j, hc in enumerate(test_hcs):
    tg = tag_of(hc)
    for i, row in enumerate(fig_rows):
        ax = axes[i, j]
        m, t = single_maps[tg] if row == SINGLE_ROW else maps_by_hc[tg][row]
        if m.size and t.size:
            im = ax.imshow(m.T, origin="lower", aspect="auto", vmin=0, vmax=vmax,
                           extent=[t.min(), t.max(), LAT_BINS[0], LAT_BINS[-1]],
                           cmap="magma")
        ax.axhline(5,  color="c", ls=":", lw=0.8)
        ax.axhline(40, color="c", ls=":", lw=0.8)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        if i == 0:
            ax.set_title(tg, fontsize=15, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{YLAB[row]}\n|lat| (°)", fontsize=12)
        if i == nrows - 1:
            ax.set_xlabel("year", fontsize=13)
        ax.tick_params(labelsize=11)
        # metric annotation on the scored model rows (not empirical / single-draw)
        if row in metrics_by_hc[tg]:
            mm = metrics_by_hc[tg][row]
            ax.text(0.03, 0.97,
                    f"NLL {mm['nll']:.2f}\nEMD {mm['emd']:.2f}\n"
                    f"en {mm['energy']:.2f}\ncrps_mu {mm['crps_mu']:.2f}",
                    transform=ax.transAxes, va="top", ha="left", fontsize=10,
                    color="white",
                    bbox=dict(boxstyle="round,pad=0.2", fc="black", ec="none", alpha=0.5))

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), aspect=40, pad=0.01)
cbar.set_label("density", fontsize=13)
cbar.ax.tick_params(labelsize=11)
fig.suptitle(f"Test-set butterfly diagrams — empirical vs SARGE vs classical vs "
             f"AI empirical-target ({SHOWCASE_NAME}, w={EMP_W:.1f})\n"
             f"(AI row = mean of {K} draws; bottom row = one fresh random draw)",
             fontsize=16)
try:
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.04)
except Exception:
    pass
plt.show()

---
## Area-weighted view — density weighted across the whole cycle

The grid above normalizes every 6-month window independently, so cycle minimum and cycle
maximum look equally bright. Here we instead weight each window's latitude density by the
**smoothed total sunspot area** at that window (`area_smoothed` from the conditioning
parquet) and renormalize each hemicycle so the whole cycle integrates to 1. The panels now
show *where the activity actually concentrates*: the cycle-maximum windows dominate, the way
the empirical butterfly's bright core does. The same area weight is applied to all rows
so the comparison stays apples-to-apples.

In [ ]:
from matplotlib.colors import PowerNorm
from matplotlib.ticker import MaxNLocator
import time

SINGLE_ROW = f"{AI_ROW} (single draw)"
YLAB = {"Empirical": "Empirical", "SARGE": "SARGE", "Classical": "Classical",
        AI_ROW: "AI (empirical)",
        SINGLE_ROW: "AI (empirical)\n(single draw)"}
# Fresh single reverse-diffusion realization every run (bottom row).
torch.manual_seed(time.time_ns() % (2**31))
single_maps = {tag_of(hc): single_emp_map(hc) for hc in test_hcs}

# Per-window weight = smoothed total sunspot area from the conditioning parquet.
_t0_area = {(int(c), str(h)): float(classical.lookup_t0(int(c), h))
            for (c, h) in classical.known_hemicycles()}
_area_recs = {}
for _, r in windows_v2_full.iterrows():
    key = (int(r["cycle"]), str(r["hemisphere"]))
    if key in _t0_area:
        _area_recs.setdefault(key, []).append(
            (float(r["tau_center"]) + _t0_area[key], float(r["area_smoothed"])))

def area_weight(cyc, hemi, decimal_year):
    """Smoothed total sunspot area (MSH) nearest to a window center, or nan."""
    arr = _area_recs.get((int(cyc), str(hemi)))
    if not arr:
        return np.nan
    yrs  = np.array([a[0] for a in arr])
    vals = np.array([a[1] for a in arr])
    return float(vals[np.argmin(np.abs(yrs - decimal_year))])

def weighted_map(hc, m, t):
    """Area-weight each window column, then renormalize the hemicycle to unit mass."""
    if not (m.size and t.size):
        return m
    w = np.array([area_weight(hc["cycle"], hc["hemisphere"], float(tt)) for tt in t])
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    wm = m * w[:, None]
    tot = wm.sum()
    return wm / tot if tot > 0 else wm

# Precompute weighted maps + a shared color scale.
area_rows = MODEL_ROWS + [SINGLE_ROW]
wmaps_by_hc, _allvals = {}, []
for hc in test_hcs:
    tg = tag_of(hc)
    wmaps_by_hc[tg] = {}
    for row in area_rows:
        m, t = single_maps[tg] if row == SINGLE_ROW else maps_by_hc[tg][row]
        wm = weighted_map(hc, m, t)
        wmaps_by_hc[tg][row] = (wm, t)
        if wm.size:
            _allvals.append(wm.ravel())
# The weighted density is heavily right-skewed (a few bright cycle-max cells,
# most cells ~0). A linear scale would clip/saturate that core, so we use the
# TRUE global max (no clipping) with a gamma=0.5 PowerNorm to lift the mid-tones
# and reveal the full wing structure. Colorbar is therefore perceptual, not linear.
gmax_w = float(np.concatenate(_allvals).max()) if _allvals else 1.0
_norm_w = PowerNorm(gamma=0.5, vmin=0.0, vmax=gmax_w)

nrows, ncols = len(area_rows), len(test_hcs)
# Column widths proportional to each hemicycle's duration (years), so every
# x axis shares one year scale.
col_spans = [max(float(np.ptp(wmaps_by_hc[tag_of(hc)]["Classical"][1])), 0.5)
             if wmaps_by_hc[tag_of(hc)]["Classical"][1].size else 1.0
             for hc in test_hcs]
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(0.55 * sum(col_spans) + 2.0, 2.7 * nrows),
                         sharex="col", sharey=True, squeeze=False,
                         gridspec_kw={"width_ratios": col_spans},
                         layout="constrained")
im = None
for j, hc in enumerate(test_hcs):
    tg = tag_of(hc)
    for i, row in enumerate(area_rows):
        ax = axes[i, j]
        wm, t = wmaps_by_hc[tg][row]
        if wm.size and t.size:
            im = ax.imshow(wm.T, origin="lower", aspect="auto", norm=_norm_w,
                           extent=[t.min(), t.max(), LAT_BINS[0], LAT_BINS[-1]],
                           cmap="magma")
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        if i == 0:
            ax.set_title(tg, fontsize=15, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{YLAB[row]}\n|lat| (°)", fontsize=12)
        if i == nrows - 1:
            ax.set_xlabel("year", fontsize=13)
        ax.tick_params(labelsize=11)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), aspect=40, pad=0.01)
cbar.set_label("cycle-weighted density (γ=0.5 scale)", fontsize=13)
cbar.ax.tick_params(labelsize=11)
fig.suptitle("Area-weighted test-set butterfly diagrams — each window scaled by smoothed "
             "total sunspot area (area_smoothed), then renormalized per hemicycle\n"
             f"(AI = {SHOWCASE_NAME}, w={EMP_W:.1f}; mean of {K} draws; bottom row = one "
             "fresh random draw)", fontsize=16)
try:
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.04)
except Exception:
    pass
plt.show()

---
## Butterfly realizations — sampled groups over the density

Same 5×6 layout and area-weighted (γ=0.5) density background as above, but each panel now
also shows **one realization**: for every 6-month window we draw the **same number of groups
that were observed** and place them at the **real observation dates**. The **Empirical** row
shows the actual observed groups; **SARGE** / **Classical** draw latitudes from their
continuous N(μ, σ); the two **AI** rows draw from their 15-bin softmax densities — the
**mean-of-K** density (4th row) and one fresh **single-draw** density (bottom row), the two
different generated distributions. Re-run the cell to resample everything.

In [ ]:
# ---- Butterfly realizations (scatter) over the area-weighted density ----
# Overlay: each panel shows its row's area-weighted probability field (faded) with
# ONE realization on top — the same number of groups as were OBSERVED in each
# 6-month window, placed at the real observation dates. Empirical shows the actual
# observed groups; SARGE/Classical draw from their continuous N(mu, sigma); the two
# AI rows draw from their 15-bin softmax densities — the mean-of-K density
# (2nd-from-bottom) and one fresh single-draw density (bottom). Re-run to resample.
import time
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import PowerNorm

rng = np.random.default_rng()                       # fresh draws every run
torch.manual_seed(time.time_ns() % (2**31))

SINGLE_ROW = f"{AI_ROW} (single draw)"
scatter_rows = MODEL_ROWS + [SINGLE_ROW]
YLAB = {"Empirical": "Empirical", "SARGE": "SARGE", "Classical": "Classical",
        AI_ROW: "AI (empirical)", SINGLE_ROW: "AI (empirical)\n(single draw)"}
# PT = dict(s=7, marker="o", linewidths=0, color="#19e6ff", alpha=0.45)   # low-alpha scatter
PT = dict(s=7, marker="o", linewidths=0, color="#ffffff", alpha=0.45)   # low-alpha scatter

# Fresh single reverse-diffusion realization -> the bottom row's distribution.
single_maps = {tag_of(hc): single_emp_map(hc) for hc in test_hcs}

# Raw catalog for true observation dates (same filtering as build_eval_hemicycles).
_raw = pd.read_csv(paths["raw_csv"])
_raw["date"]         = pd.to_datetime(_raw[["year", "month", "day"]])
_raw = _raw.dropna(subset=["CYCLE"]).copy()
_raw["CYCLE"]        = _raw["CYCLE"].astype(int)
_raw["abs_lat"]      = _raw["latitude"].abs()
_raw["hemisphere"]   = np.where(_raw["latitude"] >= 0, "north", "south")
_raw["decimal_year"] = _raw["date"].dt.year + _raw["date"].dt.dayofyear / 365.25

def observed_groups(cyc, hemi, center_decimal):
    """(times, |lat|) of observed groups in the 6-month window around a center."""
    yr = int(np.floor(center_decimal))
    if center_decimal - yr < 0.5:
        ws, we = pd.Timestamp(yr, 1, 1), pd.Timestamp(yr, 7, 1)
    else:
        ws, we = pd.Timestamp(yr, 7, 1), pd.Timestamp(yr + 1, 1, 1)
    sel = _raw[(_raw["CYCLE"] == int(cyc)) & (_raw["hemisphere"] == hemi)
               & (_raw["date"] >= ws) & (_raw["date"] < we)]
    return sel["decimal_year"].to_numpy(), sel["abs_lat"].to_numpy()

def _draw_bins(dens, n):
    """Draw n latitudes from a 15-bin density (bin center + uniform jitter)."""
    p = np.maximum(np.asarray(dens, float), 0.0)
    p = p / p.sum() if p.sum() > 0 else np.full(len(BIN_CENTERS), 1.0 / len(BIN_CENTERS))
    idx = rng.choice(len(BIN_CENTERS), size=n, p=p)
    return np.clip(BIN_CENTERS[idx] + rng.uniform(-BIN_WIDTH / 2, BIN_WIDTH / 2, n), 0, 45)

# Area-weighted (gamma=0.5) density backgrounds + one realization per (row, hemicycle).
bg_by_hc, pts_by_hc, _bgvals = {}, {}, []
for hc in test_hcs:
    tg = tag_of(hc)
    t_ref = maps_by_hc[tg]["Classical"][1]
    blk_by_t = {float(b["center_decimal"]): b for b in hc["blocks"]}
    dmaps = {"Empirical": maps_by_hc[tg]["Empirical"][0],
             "SARGE":     maps_by_hc[tg]["SARGE"][0],
             "Classical": maps_by_hc[tg]["Classical"][0],
             AI_ROW:      maps_by_hc[tg][AI_ROW][0],
             SINGLE_ROW:  single_maps[tg][0]}
    bg_by_hc[tg] = {}
    for row, dm in dmaps.items():
        wm = weighted_map(hc, dm, t_ref)
        bg_by_hc[tg][row] = (wm, t_ref)
        if wm.size:
            _bgvals.append(wm.ravel())

    rows_pts = {r: ([], []) for r in scatter_rows}
    for i, t in enumerate(t_ref):
        blk = blk_by_t[float(t)]
        times, obs_lats = observed_groups(hc["cycle"], hc["hemisphere"], float(t))
        n = len(times)
        if n == 0:
            continue
        A = hc["amplitude"]
        _, mu_s, sig_s = sarge_bin_density(float(t), hc["cycle"], hc["hemisphere"])
        mu_c = float(classical.mu(blk["tau"])); sig_c = float(classical.sigma(mu_c, A))
        draws = {
            "Empirical": obs_lats,
            "SARGE":     np.clip(rng.normal(mu_s, sig_s, n), 0, 45),
            "Classical": np.clip(rng.normal(mu_c, sig_c, n), 0, 45),
            AI_ROW:      _draw_bins(maps_by_hc[tg][AI_ROW][0][i], n),
            SINGLE_ROW:  _draw_bins(single_maps[tg][0][i], n),
        }
        for r in scatter_rows:
            rows_pts[r][0].append(times); rows_pts[r][1].append(draws[r])
    pts_by_hc[tg] = {r: (np.concatenate(xy[0]) if xy[0] else np.array([]),
                         np.concatenate(xy[1]) if xy[1] else np.array([]))
                     for r, xy in rows_pts.items()}

gmax_w = float(np.concatenate(_bgvals).max()) if _bgvals else 1.0
_norm_w = PowerNorm(gamma=0.5, vmin=0.0, vmax=gmax_w)

nrows, ncols = len(scatter_rows), len(test_hcs)
col_spans = [max(float(np.ptp(maps_by_hc[tag_of(hc)]["Classical"][1])), 0.5)
             if maps_by_hc[tag_of(hc)]["Classical"][1].size else 1.0
             for hc in test_hcs]
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(0.55 * sum(col_spans) + 2.0, 2.7 * nrows),
                         sharex="col", sharey=True, squeeze=False,
                         gridspec_kw={"width_ratios": col_spans},
                         layout="constrained")
im = None
for j, hc in enumerate(test_hcs):
    tg = tag_of(hc)
    for i, row in enumerate(scatter_rows):
        ax = axes[i, j]
        ax.set_facecolor("black")   # so faded density keeps its dark end and the dots pop
        wm, t = bg_by_hc[tg][row]
        if wm.size and t.size:
            im = ax.imshow(wm.T, origin="lower", aspect="auto", norm=_norm_w, alpha=0.85,
                           extent=[t.min(), t.max(), LAT_BINS[0], LAT_BINS[-1]], cmap="magma")
        px, py = pts_by_hc[tg][row]
        if px.size:
            ax.scatter(px, py, **PT)
        ax.set_ylim(LAT_BINS[0], LAT_BINS[-1])
        if t.size:
            ax.set_xlim(t.min() - 0.3, t.max() + 0.3)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        if i == 0:
            ax.set_title(tg, fontsize=15, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{YLAB[row]}\n|lat| (°)", fontsize=12)
        if i == nrows - 1:
            ax.set_xlabel("year", fontsize=13)
        ax.tick_params(labelsize=11)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), aspect=40, pad=0.01)
cbar.set_label("cycle-weighted density (γ=0.5 scale)", fontsize=13)
cbar.ax.tick_params(labelsize=11)
fig.suptitle("Butterfly realizations — one draw of the observed # of groups per window "
             "(dots, at real observation dates) over each row's area-weighted density\n"
             f"(SARGE/Classical ~ N(μ,σ); AI rows sample the mean-of-{K} vs a single-draw "
             "density — re-run to resample)", fontsize=16)
try:
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.04)
except Exception:
    pass
plt.show()

---
## Realizations vs the observed density (common backdrop)

Identical to the previous figure, but **every** row's backdrop is the **observed (empirical)** area-weighted density — so each model's realization (dots) is read against the same real butterfly field. Re-run to resample.

In [ ]:
# ---- Realizations over a COMMON background: the OBSERVED (empirical) density ----
# Same per-row draws as the previous figure, but every row's backdrop is now the
# observed area-weighted density, so each model's realization is read against the
# same empirical field. Re-run to resample. (Reuses observed_groups / _draw_bins /
# _raw / weighted_map defined in the previous cell.)
import time
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import PowerNorm

rng = np.random.default_rng()
torch.manual_seed(time.time_ns() % (2**31))

SINGLE_ROW = f"{AI_ROW} (single draw)"
scatter_rows = MODEL_ROWS + [SINGLE_ROW]
YLAB = {"Empirical": "Empirical", "SARGE": "SARGE", "Classical": "Classical",
        AI_ROW: "AI (empirical)", SINGLE_ROW: "AI (empirical)\n(single draw)"}
PT = dict(s=7, marker="o", linewidths=0, color="#19e6ff", alpha=0.45)

single_maps = {tag_of(hc): single_emp_map(hc) for hc in test_hcs}

# Common background = the OBSERVED (empirical) area-weighted density per hemicycle.
emp_bg, pts_by_hc, _bgvals = {}, {}, []
for hc in test_hcs:
    tg = tag_of(hc)
    t_ref = maps_by_hc[tg]["Classical"][1]
    blk_by_t = {float(b["center_decimal"]): b for b in hc["blocks"]}
    wm = weighted_map(hc, maps_by_hc[tg]["Empirical"][0], t_ref)
    emp_bg[tg] = (wm, t_ref)
    if wm.size:
        _bgvals.append(wm.ravel())

    rows_pts = {r: ([], []) for r in scatter_rows}
    for i, t in enumerate(t_ref):
        blk = blk_by_t[float(t)]
        times, obs_lats = observed_groups(hc["cycle"], hc["hemisphere"], float(t))
        n = len(times)
        if n == 0:
            continue
        A = hc["amplitude"]
        _, mu_s, sig_s = sarge_bin_density(float(t), hc["cycle"], hc["hemisphere"])
        mu_c = float(classical.mu(blk["tau"])); sig_c = float(classical.sigma(mu_c, A))
        draws = {
            "Empirical": obs_lats,
            "SARGE":     np.clip(rng.normal(mu_s, sig_s, n), 0, 45),
            "Classical": np.clip(rng.normal(mu_c, sig_c, n), 0, 45),
            AI_ROW:      _draw_bins(maps_by_hc[tg][AI_ROW][0][i], n),
            SINGLE_ROW:  _draw_bins(single_maps[tg][0][i], n),
        }
        for r in scatter_rows:
            rows_pts[r][0].append(times); rows_pts[r][1].append(draws[r])
    pts_by_hc[tg] = {r: (np.concatenate(xy[0]) if xy[0] else np.array([]),
                         np.concatenate(xy[1]) if xy[1] else np.array([]))
                     for r, xy in rows_pts.items()}

gmax_w = float(np.concatenate(_bgvals).max()) if _bgvals else 1.0
_norm_w = PowerNorm(gamma=0.5, vmin=0.0, vmax=gmax_w)

nrows, ncols = len(scatter_rows), len(test_hcs)
col_spans = [max(float(np.ptp(maps_by_hc[tag_of(hc)]["Classical"][1])), 0.5)
             if maps_by_hc[tag_of(hc)]["Classical"][1].size else 1.0
             for hc in test_hcs]
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(0.55 * sum(col_spans) + 2.0, 2.7 * nrows),
                         sharex="col", sharey=True, squeeze=False,
                         gridspec_kw={"width_ratios": col_spans},
                         layout="constrained")
im = None
for j, hc in enumerate(test_hcs):
    tg = tag_of(hc)
    wm, t = emp_bg[tg]
    for i, row in enumerate(scatter_rows):
        ax = axes[i, j]
        ax.set_facecolor("black")
        if wm.size and t.size:
            im = ax.imshow(wm.T, origin="lower", aspect="auto", norm=_norm_w, alpha=0.85,
                           extent=[t.min(), t.max(), LAT_BINS[0], LAT_BINS[-1]], cmap="magma")
        px, py = pts_by_hc[tg][row]
        if px.size:
            ax.scatter(px, py, **PT)
        ax.set_ylim(LAT_BINS[0], LAT_BINS[-1])
        if t.size:
            ax.set_xlim(t.min() - 0.3, t.max() + 0.3)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        if i == 0:
            ax.set_title(tg, fontsize=15, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{YLAB[row]}\n|lat| (°)", fontsize=12)
        if i == nrows - 1:
            ax.set_xlabel("year", fontsize=13)
        ax.tick_params(labelsize=11)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), aspect=40, pad=0.01)
cbar.set_label("observed cycle-weighted density (γ=0.5 scale)", fontsize=13)
cbar.ax.tick_params(labelsize=11)
fig.suptitle("Realizations vs the OBSERVED density — every row's backdrop is the observed "
             "(empirical) area-weighted density;\ndots are each model's draw of the observed "
             "# of groups per window at real dates (re-run to resample)", fontsize=16)
try:
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.04)
except Exception:
    pass
plt.show()

---
## Test scorecard — does the NLL agree with the eye?

Metrics pooled over all 6 test hemicycles, one row per model. Lower is better for every
column. The empirical distribution is the reference and is omitted from the table.

The NLL column is the renormalized (proper-density) hard NLL under the identical
`μ ≤ μ₀(A)` gate for all three models, so this table is directly comparable to 11c's —
same rows, same blocks, only the AI row's model changes.

The physical checks below score the *assembled* butterflies rather than the density at
observed latitudes: Spörer slope (should be negative — equatorward drift), and the
fraction of mass inside the 5°–40° Spörer band.

In [ ]:
# Pool the per-window density dicts across all 6 test hemicycles, then score once.
emp_all, tau_all = {}, {}
model_all = {"SARGE": {}, "Classical": {}, AI_ROW: {}}
for tg, pl in pooled_by_hc.items():
    emp_all.update(pl["emp"]); tau_all.update(pl["tau"])
    for mdl in model_all:
        model_all[mdl].update(pl[mdl])

# Pooled renormalized NLL.
_zero_all = {(hc["cycle"], hc["hemisphere"], float(b["center_decimal"])): np.zeros(15)
             for hc in test_hcs for b in hc["blocks"]}
nll_cl_all = hard_nll_combined_normalized(classical, test_hcs, _zero_all,
                                          BIN_CENTERS, bin_width=BIN_WIDTH)[0]
_ai_keys, _ai_samp = [], []
for hc in test_hcs:
    pl = pooled_by_hc[tag_of(hc)]
    _ai_keys += list(pl["ai_keys"]); _ai_samp.append(pl["ai_samp"])
_ai_samp = np.concatenate(_ai_samp, axis=0)
nll_ai_all = k_run_combined(nll_norm_direct, classical, test_hcs,
                            _ai_keys, _ai_samp)[0].mean()

def _sarge_nll_all():
    tot, inc, eps = 0.0, 0, 1e-6
    for hc in test_hcs:
        A, mu0 = hc["amplitude"], float(classical.mu_0(hc["amplitude"]))
        for b in hc["blocks"]:
            if float(classical.mu(b["tau"])) > mu0:
                continue
            dens = np.maximum(eps, sarge_bin_density(float(b["center_decimal"]),
                                                     hc["cycle"], hc["hemisphere"])[0])
            Z = max(eps, float((dens * BIN_WIDTH).sum()))
            bin_ix = np.clip(np.floor(b["lats"] / BIN_WIDTH).astype(int), 0, 14)
            ll = np.log(dens[bin_ix]).mean()
            if np.isfinite(ll):
                tot -= (ll - np.log(Z)); inc += 1
    return tot / inc if inc else float("nan")

nll_all = {"SARGE": _sarge_nll_all(), "Classical": nll_cl_all,
           AI_ROW: float(nll_ai_all)}

rows = []
for mdl in ["SARGE", "Classical", AI_ROW]:
    rep = distributional_scorecard(model_all[mdl], emp_all, BIN_CENTERS, BIN_WIDTH, tau_all)
    rows.append({"model": mdl, "nll": nll_all[mdl],
                 "emd": rep["emd"], "energy": rep["energy"], "crps_mu": rep["crps_mu"],
                 "mu_mae": rep["mu_mae"], "sigma_mae": rep["sigma_mae"],
                 "n_blocks": rep["n_blocks"]})
scorecard_test = pd.DataFrame(rows).set_index("model")
print(f"Test-set scorecard — AI = {SHOWCASE_NAME} (w={EMP_W:.1f}), "
      f"pooled over 6 hemicycles; lower is better:\n")
print(scorecard_test.to_string(float_format=lambda v: f"{v:.4f}"))

# Physical plausibility of the assembled butterflies, per hemicycle.
print("\nPhysical checks (Spörer slope deg/yr; in-band = mass fraction in 5–40°):")
phys_rows = []
for row in MODEL_ROWS:
    for hc in test_hcs:
        tg = tag_of(hc)
        m, t = maps_by_hc[tg][row]
        if m.shape[0] < 2:
            continue
        c = butterfly_physical_checks(m, t, BIN_CENTERS)
        phys_rows.append({"model": row, "hemicycle": tg,
                          "sporer_slope": c["sporer_slope"],
                          "sporer_ok": c["sporer_ok"],
                          "in_band": c["in_band_fraction"]})
phys_test = pd.DataFrame(phys_rows)
print(phys_test.pivot(index="hemicycle", columns="model", values="sporer_slope")
      .reindex(columns=MODEL_ROWS).to_string(float_format=lambda v: f"{v:+.3f}"))
print("\nmean in-band fraction (ideal ~ empirical row):")
print(phys_test.groupby("model")["in_band"].mean().reindex(MODEL_ROWS)
      .to_string(float_format=lambda v: f"{v:.3f}"))
scorecard_test

---
## Held-out ranking of every trained empirical variant

The figures above showcase one variant. This cell scores **every** `ckpt_emp_*.ckpt` that
has an `EXPERIMENTS_EMP` spec on the same 6 test hemicycles, with the classical and SARGE
baselines in the same table, so the whole ablation gets a held-out verdict in one place.

`K_TABLE` is smaller than the showcase `K` to keep the sweep tractable; the NLL column is
the mean over the K draws (same estimator as the showcase), so the showcased variant's row
here will differ from the scorecard above only by Monte-Carlo noise.

CFG-trained variants (`cond_dropout_p > 0`) are scored at every weight in `CFG_W_TEST` —
default unguided only. Extend the list to sweep guidance on the test set.

**This table is a report, not a model-selection tool.** Pick your variant on val (11d);
reading the winner off this table and then quoting its test number is test-set selection.

In [ ]:
K_TABLE    = 100
CFG_W_TEST = [0.0]          # e.g. [0.0, 1.0, 2.0] to sweep guidance on test

def _score_variant_on_test(name, cfg, w):
    """Pooled test NLL + distributional + physical metrics for one variant."""
    lit, train_ds, _, _ = load_trained_empirical_experiment(
        name=name, cfg=cfg, windows_aug=windows_v2_full, classical=classical,
        bin_centers=BIN_CENTERS, ckpt_dir=CKPT_DIR,
        alpha_np=alpha_np, sigma_np=sigma_np, bin_width=BIN_WIDTH,
    )
    all_keys, all_samp = [], []
    model_dens, emp_dens, tau_by = {}, {}, {}
    slopes, in_band = [], []
    for hc in test_hcs:
        keys, cond = block_cond_concat([hc], lit, cfg, train_ds)
        if not keys:
            continue
        torch.manual_seed(0)
        samp = sample_empirical_extended(
            lit, cond.repeat_interleave(K_TABLE, dim=0), guidance_w=w,
            bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy().reshape(len(keys), K_TABLE, 15)
        _simplex_sanity(samp)
        all_keys += list(keys); all_samp.append(samp)

        mean_dens = {k: samp[i].mean(axis=0) for i, k in enumerate(keys)}
        m, t = assemble_butterfly_direct(classical, hc, mean_dens, BIN_CENTERS)
        if m.shape[0] >= 2:
            c = butterfly_physical_checks(m, t, BIN_CENTERS)
            slopes.append(c["sporer_slope"]); in_band.append(c["in_band_fraction"])

        # Score only the gated windows, matching the pooled dicts built above.
        for key in pooled_by_hc[tag_of(hc)]["emp"]:
            i = keys.index(key)
            model_dens[key] = samp[i]
            emp_dens[key]   = pooled_by_hc[tag_of(hc)]["emp"][key]
            tau_by[key]     = pooled_by_hc[tag_of(hc)]["tau"][key]

    all_samp = np.concatenate(all_samp, axis=0)
    nlls, _  = k_run_combined(nll_norm_direct, classical, test_hcs, all_keys, all_samp)
    rep = distributional_scorecard(model_dens, emp_dens, BIN_CENTERS, BIN_WIDTH, tau_by)
    return {"experiment": name, "guidance_w": w,
            "nll_mean": float(nlls.mean()), "nll_std": float(nlls.std()),
            "emd": rep["emd"], "energy": rep["energy"], "crps_mu": rep["crps_mu"],
            "mu_mae": rep["mu_mae"], "sigma_mae": rep["sigma_mae"],
            "sporer_slope": float(np.mean(slopes)) if slopes else np.nan,
            "in_band": float(np.mean(in_band)) if in_band else np.nan,
            "n_blocks": rep["n_blocks"]}

# Baseline rows (deterministic; guidance not applicable).
_base_rows = []
for mdl in ["SARGE", "Classical", "Empirical"]:
    if mdl == "Empirical":
        _sl = [butterfly_physical_checks(*maps_by_hc[tag_of(hc)]["Empirical"][:2],
                                         BIN_CENTERS)
               for hc in test_hcs if maps_by_hc[tag_of(hc)]["Empirical"][0].shape[0] >= 2]
        _base_rows.append({"experiment": "empirical (reference)", "guidance_w": np.nan,
                           "nll_mean": np.nan, "nll_std": np.nan,
                           "emd": 0.0, "energy": 0.0, "crps_mu": np.nan,
                           "mu_mae": 0.0, "sigma_mae": 0.0,
                           "sporer_slope": float(np.mean([c["sporer_slope"] for c in _sl])),
                           "in_band": float(np.mean([c["in_band_fraction"] for c in _sl])),
                           "n_blocks": len(emp_all)})
        continue
    rep = distributional_scorecard(model_all[mdl], emp_all, BIN_CENTERS, BIN_WIDTH, tau_all)
    _sl = [butterfly_physical_checks(*maps_by_hc[tag_of(hc)][mdl][:2], BIN_CENTERS)
           for hc in test_hcs if maps_by_hc[tag_of(hc)][mdl][0].shape[0] >= 2]
    _base_rows.append({"experiment": mdl.lower(), "guidance_w": np.nan,
                       "nll_mean": nll_all[mdl], "nll_std": 0.0,
                       "emd": rep["emd"], "energy": rep["energy"],
                       "crps_mu": rep["crps_mu"], "mu_mae": rep["mu_mae"],
                       "sigma_mae": rep["sigma_mae"],
                       "sporer_slope": float(np.mean([c["sporer_slope"] for c in _sl])),
                       "in_band": float(np.mean([c["in_band_fraction"] for c in _sl])),
                       "n_blocks": rep["n_blocks"]})

variant_rows = list(_base_rows)
for name in KNOWN_EMP:
    cfg = EXPERIMENTS_EMP[name]
    ws  = CFG_W_TEST if cfg.get("cond_dropout_p", 0.0) > 0.0 else [0.0]
    for w in ws:
        print(f"scoring {name} (w={w}) on test ...")
        # Load from the on-disk stem; the row is labeled with the canonical name.
        _row = _score_variant_on_test(EMP_STEM[name], cfg, w)
        _row["experiment"] = name
        variant_rows.append(_row)

test_ranking = pd.DataFrame(variant_rows)
print(f"\nHeld-out test ranking (K={K_TABLE} draws/window, 6 hemicycles). "
      f"Lower nll/emd/energy/crps_mu/*_mae is better; sporer_slope should be negative:\n")
print(test_ranking.sort_values("nll_mean").to_string(
    index=False, float_format=lambda v: f"{v:.4f}"))

---
## Reading the result

1. **Does the empirical-target model beat classical and SARGE on held-out data?**
   Compare the `AI (empirical)` row of the scorecard against `Classical` and `SARGE` —
   on NLL *and* on the distributional metrics. Disagreement between the two is
   informative: NLL is pointwise and tail-dominated, EMD/energy/crps_mu track shape.

2. **Does it stay physical off the training set?** The Spörer slopes should be negative
   for every test hemicycle and the in-band fraction should sit near the empirical row's.
   A model that wins on NLL while drifting poleward or leaking mass outside 5°–40° has
   not earned the pipeline slot.

3. **Empirical target vs residual target.** The gate, the blocks and the NLL convention
   are identical to `11c_test_showcase.ipynb`, so the two `scorecard_test` tables are
   directly subtractable — that difference is the held-out version of the framing
   question 11d answers on val.